# Sweep Analysis Template

Minimal reusable template for analysing parameter-sweep experiments.

## How to use
1. Copy this notebook to the experiment-specific folder (or just import `load_sweep_df` from here).
2. Edit the **Configuration** cell to point at your sweep root and choose the metrics file / key.
3. Run all cells – `tot` is a flat DataFrame with one row per combination and one column per sweep parameter + metric.
4. Add analysis cells below.

## Expected sweep folder structure
```
<SWEEP_ROOT>/
  combinations_data.json        # lists all combinations with their param values
  runs/
    combinations/
      <combo_name>/             # one folder per combination
        kfold_summary.json      # (or another METRICS_FILE) with aggregated metrics
        k_0/
          best_causal_metrics.json
          best_reconstruction_metrics.json
```

In [ ]:
import json
from os.path import join, exists

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---
## Configuration  ← edit this cell

In [ ]:
# Path to the sweeper folder (contains combinations_data.json and runs/)
SWEEP_ROOT = "../experiments/<experiment_folder>/sweeper"

# JSON file inside each combination folder that holds the metrics
# Common choices:
#   "kfold_summary.json"              → last-epoch aggregated metrics (statistics.*.mean)
#   "k_0/best_causal_metrics.json"    → best causal checkpoint metrics (flat dict)
#   "k_0/best_reconstruction_metrics.json"
METRICS_FILE = "kfold_summary.json"

# Top-level key inside METRICS_FILE to read metrics from.
# Use "statistics" for kfold_summary.json (gives mean/std/min/max per metric).
# Use None (or "") to read the JSON root directly (flat dict of metric → value).
METRIC_KEY = "statistics"

# When METRIC_KEY="statistics", which sub-key to extract as the representative value.
# Typically "mean".  Set to None to keep the full nested dict.
STAT_SUBKEY = "mean"

---
## Core loader

In [ ]:
def load_sweep_df(
    sweep_root: str,
    metrics_file: str = "kfold_summary.json",
    metric_key: str = "statistics",
    stat_subkey: str = "mean",
) -> pd.DataFrame:
    """
    Build a flat DataFrame with one row per sweep combination.

    Columns
    -------
    - One column per sweep parameter (e.g. 'lr', 'd_model_set', 'seed', ...).
    - One column per metric extracted from *metrics_file*.
    - 'combo_name'  : folder name of the combination.
    - 'combo_dir'   : full path to the combination folder.
    - 'missing_data': True when metrics_file was not found.

    Parameters
    ----------
    sweep_root   : path to sweeper folder (contains combinations_data.json).
    metrics_file : filename (relative to combo folder) with the metrics.
    metric_key   : top-level key inside the JSON, or "" / None for root.
    stat_subkey  : if metric_key yields a dict-of-dicts (e.g. statistics),
                   extract this sub-key (e.g. 'mean') as the scalar value.
                   Set to None to keep the nested structure.
    """
    combinations_path = join(sweep_root, "combinations_data.json")
    runs_dir = join(sweep_root, "runs", "combinations")

    with open(combinations_path) as f:
        combo_data = json.load(f)

    rows = []
    for combo in combo_data["combinations"]:
        name = combo["name"]
        params = combo["params"]           # dict: param_name → value
        combo_dir = join(runs_dir, name)
        metrics_path = join(combo_dir, metrics_file)

        row = {"combo_name": name, "combo_dir": combo_dir, "missing_data": False}
        row.update(params)                 # add sweep parameter columns

        if not exists(metrics_path):
            row["missing_data"] = True
            rows.append(row)
            continue

        with open(metrics_path) as f:
            metrics_json = json.load(f)

        # Navigate to the desired sub-section
        if metric_key:
            metrics_section = metrics_json.get(metric_key, {})
        else:
            metrics_section = metrics_json

        # Flatten metrics
        for metric_name, metric_value in metrics_section.items():
            if isinstance(metric_value, dict) and stat_subkey is not None:
                # e.g. {"mean": 0.1, "std": 0.01, ...}  →  take stat_subkey
                row[metric_name] = metric_value.get(stat_subkey)
            else:
                row[metric_name] = metric_value

        rows.append(row)

    df = pd.DataFrame(rows)

    n_missing = df["missing_data"].sum()
    if n_missing:
        print(f"Warning: {n_missing}/{len(df)} combinations have missing metrics files.")

    return df

---
## Load

In [ ]:
tot = load_sweep_df(
    sweep_root=SWEEP_ROOT,
    metrics_file=METRICS_FILE,
    metric_key=METRIC_KEY,
    stat_subkey=STAT_SUBKEY,
)

print(f"Loaded {len(tot)} combinations.")
print(f"Columns: {tot.columns.tolist()}")
display(tot.head())

---
## Analysis

Add experiment-specific analysis cells below.